In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df=pd.read_csv('src\claims_train.csv')

df = df.drop(columns=["IDpol"])
df.drop('Area', axis=1, inplace=True)

df['BonusMalus']=df['BonusMalus']/100

df= df[df['Exposure']<=1]
df = df[(df['VehAge']<=25)]
df['Density_scaled'] = StandardScaler().fit_transform(df[['Density']])

age_avg = df[df['BonusMalus']>=0.95**(df['DrivAge']-18)].groupby('DrivAge')['BonusMalus'].mean()
df.loc[df['BonusMalus']<0.95**(df['DrivAge']-18), 'BonusMalus'] = df.loc[df['BonusMalus']<0.95**(df['DrivAge']-18), 'DrivAge'].map(age_avg)

alpha=2 # If we want to penaltize the claimnb more
beta=0 # If we want to finetune the exposure part 
gamma=0.1 # To avoid log(0)
df['Risk'] = (np.log(1+(gamma+df['ClaimNb']**alpha)/(df['Exposure']+beta))/(1+(np.log(1+(gamma+df['ClaimNb']**alpha)/(df['Exposure']+beta)))))

In [3]:
X = df.drop(columns=["ClaimNb", "Exposure", "Risk"])
y = df["Risk"]

In [4]:
X = pd.get_dummies(X, drop_first=True)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5, scoring="r2")
print(rf.get_params())
print(f"Cross-validated R² scores: {cv_scores}")
print(f"Mean R²: {cv_scores.mean():.3f}")

{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': None, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Cross-validated R² scores: [0.01198551 0.01644964 0.01278377 0.01554561 0.00990456]
Mean R²: 0.013


In [8]:
rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)

In [9]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test MSE vs formula: {mse:.4f}")
print(f"Test R² vs formula: {r2:.4f}")

Test MSE vs formula: 0.0313
Test R² vs formula: -0.0002
